Parte 3 - Cruce por ubigeo y análisis para el periodo del 1 de enero al 31 de mayo de 2025



1. Crear ubigeos.csv

In [43]:
import pandas as pd

ubigeos = pd.DataFrame({

    "ubigeo" : ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10",
                 "11", "12", "13", "14", "15", "16", "17", "18", "19", "20", "21", "22", "23", "24", "25"],
    "departamento": ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho", "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco", 
                     "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto", "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno", 
                     "San Martín", "Tacna", "Tumbes", "Ucayali"]
})

ubigeos.to_csv("datos/ubigeos.csv", index=False, encoding="utf-8")

ubigeos

,ubigeo,departamento
0,01,Amazonas
1,02,Áncash
2,03,Apurímac
3,04,Arequipa
4,05,Ayacucho
5,06,Cajamarca
6,07,Callao
7,08,Cusco
8,09,Huancavelica
9,10,Huánuco


2. Leer dos veces ubigeos.csv y mostrar el resultado de la comparación

In [44]:
ubigeos1 = pd.read_csv("datos/ubigeos.csv")
ubigeos2 = pd.read_csv("datos/ubigeos.csv", dtype={"ubigeo": str})

print(ubigeos1.head())
print(ubigeos2.head())


   ubigeo departamento
0       1     Amazonas
1       2       Áncash
2       3     Apurímac
3       4     Arequipa
4       5     Ayacucho
  ubigeo departamento
0     01     Amazonas
1     02       Áncash
2     03     Apurímac
3     04     Arequipa
4     05     Ayacucho


En el caso de la primera forma de lectura (read_csv), se asume que el contenido de la columna ubigeo son números, por lo que elimina los ceros a la izquierda. Mientras, en la segunda forma de lectura (dtype={"ubigeo": str}) se lee el contenido de la columna ubigeos como texto, por lo que se mantienen los ceros que son parte del código. Teniendo en cuenta lo anterior, la lectura se debe hacer con la segunda forma, ya que, de lo contrario, al hacer el merge no se hará correctamente.  

3. Agregar ubigeos a decretos_por_departamento.csv y lluvias_por_departamente.csv

In [45]:
# Se abre las tablas de decretos y lluvias 

decretos = pd.read_csv("datos/decretos_por_departamento.csv")
lluvias = pd.read_csv("datos/lluvias_por_departamento.csv")

print(decretos.head())
print(lluvias.head())

  departamento  declaratorias  prorrogas
0     Amazonas              8          4
1       Áncash              8          6
2     Apurímac              7          2
3     Arequipa              8          2
4     Ayacucho              8          4
  departamento      capital   latitud  longitud  lluvia_total_mm  \
0     Amazonas  Chachapoyas  -6.23169 -77.86903            427.9   
1       Áncash       Huaraz  -9.52614 -77.52869           1415.9   
2     Apurímac      Abancay -13.63426 -72.88380            820.0   
3     Arequipa     Arequipa -16.39899 -71.53747            439.4   
4     Ayacucho     Ayacucho -13.16380 -74.22345            500.8   

   dias_lluvia_fuerte  
0                   0  
1                  11  
2                   3  
3                   3  
4                   1  


In [51]:
# merge de las tablas de decretos y lluvias con ubigeos

decretos = pd.read_csv("datos/decretos_por_departamento.csv")
lluvias = pd.read_csv("datos/lluvias_por_departamento.csv")

decretos = pd.merge(decretos, ubigeos2, on="departamento", how="left")
lluvias = pd.merge(lluvias, ubigeos2, on="departamento", how="left")

decretos.head()

,departamento,declaratorias,prorrogas,ubigeo
0,Amazonas,8,4,01
1,Áncash,8,6,02
2,Apurímac,7,2,03
3,Arequipa,8,2,04
4,Ayacucho,8,4,05


In [52]:
lluvias.head()

,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,ubigeo
0,Amazonas,Chachapoyas,-6.23169,-77.86903,427.9,0,01
1,Áncash,Huaraz,-9.52614,-77.52869,1415.9,11,02
2,Apurímac,Abancay,-13.63426,-72.88380,820.0,3,03
3,Arequipa,Arequipa,-16.39899,-71.53747,439.4,3,04
4,Ayacucho,Ayacucho,-13.16380,-74.22345,500.8,1,05


4. Verificar si hay departamentos que no encontraron su ubigeo

In [53]:
sin_ubigeo_decretos = decretos[decretos["ubigeo"].isna()]
sin_ubigeo_lluvias = lluvias[lluvias["ubigeo"].isna()]

print("Sin ubigeo en decretos:", len(sin_ubigeo_decretos))
print(sin_ubigeo_decretos["departamento"].tolist())

print("Sin ubigeo en lluvias:", len(sin_ubigeo_lluvias))
print(sin_ubigeo_lluvias["departamento"].tolist())

Sin ubigeo en decretos: 0
[]
Sin ubigeo en lluvias: 0
[]


Con lo anterior, damos cuenta que ningun departamento se quedó sin su ubigeo en ninguna de las dos tablas. 

5. Cruzar las tablas por ubigeo

In [54]:
tabla_final = pd.merge(
    ubigeos2,
    lluvias.drop(columns="departamento"),
    on="ubigeo",
    how="left",
)

tabla_final = pd.merge(
    tabla_final,
    decretos.drop(columns="departamento"),
    on="ubigeo",
    how="left",
)

print(tabla_final.shape)
tabla_final

(25, 9)


,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
0,01,Amazonas,Chachapoyas,-6.23169,-77.86903,427.9,0,8,4
1,02,Áncash,Huaraz,-9.52614,-77.52869,1415.9,11,8,6
2,03,Apurímac,Abancay,-13.63426,-72.88380,820.0,3,7,2
3,04,Arequipa,Arequipa,-16.39899,-71.53747,439.4,3,8,2
4,05,Ayacucho,Ayacucho,-13.16380,-74.22345,500.8,1,8,4
5,06,Cajamarca,Cajamarca,-7.16378,-78.50027,875.7,2,7,2
6,07,Callao,Callao,-12.05162,-77.13452,54.0,0,2,0
7,08,Cusco,Cusco,-13.53188,-71.96701,801.3,4,9,6
8,09,Huancavelica,Huancavelica,-12.78691,-74.97298,930.4,4,8,2
9,10,Huánuco,Huánuco,-9.92882,-76.23989,617.0,4,8,5


6. Verificar que la tabla final tenga 25 filas

In [55]:
print("Vacíos antes de rellenar:", tabla_final[["declaratorias", "prorrogas"]].isna().sum().sum())

# Los departamentos sin decretos quedan con 0
tabla_final["declaratorias"] = tabla_final["declaratorias"].fillna(0).astype(int)
tabla_final["prorrogas"] = tabla_final["prorrogas"].fillna(0).astype(int)

print("Filas de la tabla final:", len(tabla_final))
print("Ubigeos repetidos:", tabla_final["ubigeo"].duplicated().sum())
print("Departamentos con 0 declaratorias:", (tabla_final["declaratorias"] == 0).sum())

Vacíos antes de rellenar: 0
Filas de la tabla final: 25
Ubigeos repetidos: 0
Departamentos con 0 declaratorias: 0


7. Guardar datos/tabla_final.csv

In [57]:
# Asegurar que el ubigeo sea texto de 2 dígitos
tabla_final["ubigeo"] = tabla_final["ubigeo"].astype(str).str.zfill(2)

tabla_final.to_csv("datos/tabla_final.csv", index=False, encoding="utf-8")

# Comprobación: volver a leer el archivo guardado
revision = pd.read_csv("datos/tabla_final.csv", dtype={"ubigeo": str})
print("Filas y columnas:", revision.shape)
print("Primeros ubigeos:", revision["ubigeo"].head(10).tolist())
print("¿Todos tienen 2 dígitos?:", (revision["ubigeo"].str.len() == 2).all())

revision

Filas y columnas: (25, 9)
Primeros ubigeos: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10']
¿Todos tienen 2 dígitos?: True


,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte,declaratorias,prorrogas
0,01,Amazonas,Chachapoyas,-6.23169,-77.86903,427.9,0,8,4
1,02,Áncash,Huaraz,-9.52614,-77.52869,1415.9,11,8,6
2,03,Apurímac,Abancay,-13.63426,-72.88380,820.0,3,7,2
3,04,Arequipa,Arequipa,-16.39899,-71.53747,439.4,3,8,2
4,05,Ayacucho,Ayacucho,-13.16380,-74.22345,500.8,1,8,4
5,06,Cajamarca,Cajamarca,-7.16378,-78.50027,875.7,2,7,2
6,07,Callao,Callao,-12.05162,-77.13452,54.0,0,2,0
7,08,Cusco,Cusco,-13.53188,-71.96701,801.3,4,9,6
8,09,Huancavelica,Huancavelica,-12.78691,-74.97298,930.4,4,8,2
9,10,Huánuco,Huánuco,-9.92882,-76.23989,617.0,4,8,5


8. Hacer dos gráficos con Plotly

In [75]:
import plotly.express as px

#Grafico de dispersión: lluvia total vs. declaratorias por departamento

# Posición del nombre para los puntos que se enciman; el resto va arriba
posicion_especial = {
    "Amazonas": "bottom left",
    "Ayacucho": "bottom right",
    "Apurímac": "top left",
    "Cajamarca": "bottom center",
    "San Martín": "top right",
    "Ica": "bottom center",
    "La Libertad": "top right",
}
posiciones = tabla_final["departamento"].map(posicion_especial).fillna("top center")

fig_dispersion = px.scatter(
    tabla_final,
    x="lluvia_total_mm",
    y="declaratorias",
    text="departamento",
    title="Lluvia total vs. declaratorias de emergencia por departamento (01/01/2025 al 31/05/2025)",
    labels={
        "lluvia_total_mm": "Lluvia total (mm)",
        "declaratorias": "Declaratorias de emergencia",
    },
)
fig_dispersion.update_traces(textposition=list(posiciones), textfont_size=10)
fig_dispersion.update_layout(width=900, height=600)
fig_dispersion.show()

In [ ]:
# Grafico de barras con los 10 departamentos con más declaratorias de emergencia

In [76]:
top10 = tabla_final.sort_values(
    ["declaratorias", "lluvia_total_mm"], ascending=False
).head(10)

fig_barras = px.bar(
    top10,
    x="departamento",
    y="declaratorias",
    text="declaratorias",
    title="Los 10 departamentos con más declaratorias de emergencia (01/01/2025 al 31/05/2025)",
    labels={
        "departamento": "Departamento",
        "declaratorias": "Declaratorias de emergencia",
    },
)
fig_barras.show()

9. Respuesta a las preguntas

a) ¿Cuáles fueron los 3 departamentos con más declaratorias? ¿Coinciden con los 3 donde más llovió?

In [80]:
columnas = ["departamento", "declaratorias", "lluvia_total_mm"]

print("Más declaratorias:")
display(tabla_final.sort_values("declaratorias", ascending=False)[columnas].head(10))

print("Más lluvia:")
display(tabla_final.sort_values("lluvia_total_mm", ascending=False)[columnas].head(3))

Más declaratorias:


,departamento,declaratorias,lluvia_total_mm
7,Cusco,9,801.3
0,Amazonas,8,427.9
1,Áncash,8,1415.9
4,Ayacucho,8,500.8
3,Arequipa,8,439.4
8,Huancavelica,8,930.4
15,Loreto,8,2019.9
14,Lima,8,54.0
9,Huánuco,8,617.0
21,San Martín,7,943.3


Más lluvia:


,departamento,declaratorias,lluvia_total_mm
15,Loreto,8,2019.9
16,Madre de Dios,3,1751.3
1,Áncash,8,1415.9


El primer lugar de departamentos con más lluvias corresponde a Cusco con 9 declaratorias. En el segundo lugar hay un empate entre 8 regiones (Amazonas, Ancash, Ayacucho, Arequipa, Huancavelica, Loreto, Lima y Huánuco) con 8 declaratorias. En tercer luar se encuentra la región San Martín con 7 declaratorias. 

En cuanto a los departamentos donde más llovió, los tres primeros lugares corresponden a Loreto, Madre de Dios y Ancash con 2019.9 mm, 1751.3 mm y 1415.9 mm, respectivamente. Teniendo en cuenta ello, entre los departamentos con más declaratorias y más lluvias solo coinciden Loreto y Ancash, pero no se encuentra a Madre de Dios que pese a que fue el segundo departamento donde más llovío no aparece entre los 3 regiones con más declaratorias.  

b) ¿Hay departamentos con mucha lluvia y pocas o ninguna declaratoria, o al revés? Den al menos dos razones posibles.

In [82]:
columnas = ["departamento", "declaratorias", "lluvia_total_mm"]

print("Menos declaratorias:")
display(tabla_final.sort_values(["declaratorias", "lluvia_total_mm"])[columnas].head(10))

print("Menos lluvia:")
display(tabla_final.sort_values("lluvia_total_mm")[columnas].head(10))

Menos declaratorias:


,departamento,declaratorias,lluvia_total_mm
6,Callao,2,54.0
23,Tumbes,2,699.1
13,Lambayeque,3,128.1
16,Madre de Dios,3,1751.3
17,Moquegua,5,97.5
19,Piura,5,241.6
22,Tacna,6,17.4
10,Ica,6,124.3
12,La Libertad,6,143.8
18,Pasco,6,874.2


Menos lluvia:


,departamento,declaratorias,lluvia_total_mm
22,Tacna,6,17.4
6,Callao,2,54.0
14,Lima,8,54.0
17,Moquegua,5,97.5
10,Ica,6,124.3
13,Lambayeque,3,128.1
12,La Libertad,6,143.8
19,Piura,5,241.6
0,Amazonas,8,427.9
3,Arequipa,8,439.4


Entre los departamentos con mucha lluvia y pocas declaratorias encontramos en el caso de Madre de Dios que tuvo 1751.3 mm de lluvia y solo 3 declaratorias. Mientras entre los departamentos con poca lluvia y muchas declaratorias está Tacna con 17.4 mm de lluvia y 6 declaratorias, Lima con 54 mm y 8 declaratorias e Ica con 124.3 mm de lluvia y 6 declaratorias. 

Esta situación se podría explicar por:
- La forma en que se han recopilado los datos, ya que la cantidad de lluvia recogida en nuestra tabla solo corresponde a un punto de la región, siendo esta la capital, sin embargo, las declaratorias se pueden dar por lo que ocurrió en cualquier provincia o distrito de la región. Así, tenemos regiones como Lima, donde se reporta pocas lluvias al encontrarse la capital en una zona desértica, pero muchas declaratorias probablemente por lo que ocurre en otras provincias de la región que están en la zona sierra. 
- Los motivos bajos los que se realizan declaratorias de emergencia, pues ello no depende necesariamente de la cantidad de lluvia que cae, sino por los daños causados por la misma. Así, en zonas donde suele llover mucho es probable que estén más preparadas y por lo tanto haya menos daños, mientras en zonas donde no llueve mucho, un incremento por leve que sea puede causar daños.  

c) ¿Qué porcentaje de las declaratorias fue por peligro inminente (antes del daño) y qué porcentaje por impacto de daños (después)? ¿Qué dice eso sobre la gestión del riesgo en su temporada?

In [83]:
decretos_lluvias = pd.read_csv("datos/decretos_lluvias.csv")

declaratorias = decretos_lluvias[decretos_lluvias["tipo"] == "declara"]

print("Total de declaratorias:", len(declaratorias))
print(declaratorias["motivo"].value_counts())
print((declaratorias["motivo"].value_counts(normalize=True) * 100).round(1))

Total de declaratorias: 11
motivo
impacto de daños     9
peligro inminente    2
Name: count, dtype: int64
motivo
impacto de daños     81.8
peligro inminente    18.2
Name: proportion, dtype: float64


En la información reportada se observa que para esta temporada (01/01/25 al 31/05/25), el 81.8% de las declaratorias ocurrieron por impacto de daños, lo que implica que la gestión de riesgos durante este periodo fue principalmente reactiva, es decir cuando el daño ya estaba hecho. 

d) Mencionen al menos tres limitaciones de este análisis.

- La lluvia se midió en un solo punto del departamento (la capital) y no considerando todo el territorio. 
- La medición de una sola temporada en un solo año no nos permite generalizar los hallazgos. Probablemente en otro año habría un resultado distinto.
- Los reportes de declaratoria no dependen de la cantidad de lluvia que cae, sino que depende de otros factores. Por ello, aunque se halla encontrado correlación entre el número de declaratorias y la cantidad de lluvia, ello no necesariamente implica causalidad. 

e) Conclusión: respondan la pregunta del trabajo en 5 a 8 líneas.

En base a los hallazgos encontrados, no es posible afirmar que el Estado declara emergencia donde más llueve, ya que contamos con regiones con mucha lluvia y pocas declaraciones y viceversa. En ese sentido podemos concluir que mayor cantidad de lluvia no necesariamente implica más declaratorias de emergencia, por lo cual existen otros factores que explican dicha decisión.

En general, se observa que para el periodo estudiado (01/05/25 al 31/05/25) la gestión de riesgos ante las lluvias es más reactiva que preventiva. Por último, es importante señalar que la principal limitación del estudio es que la cantidad de lluvia se mide en un solo punto de los departamentos (la capital), de modo que no es posible hacer un análisis más exhaustivo. 